In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


import numpy as np 
import torch
from torchvision import datasets, transforms
import torch.nn.functional as F


import pandas as pd
import os
#from local_models import CaptumCNN, AE, Generator_Cifar, BasicBlock, ResNet_large_cifar, ResNet_small_cifar
from collections import OrderedDict
from matplotlib import pyplot as plt
from skimage.metrics import structural_similarity
from joblib import Parallel, delayed


c:\Users\Jasmin\anaconda3\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
c:\Users\Jasmin\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Jasmin\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
from __future__ import print_function

import argparse
import os
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim as optim
import torch.utils.data
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torch.nn.functional as F


class CNN(nn.Module):
	
	def __init__(self):
		super(CNN, self).__init__()
		self.main = nn.Sequential(
			
			# input is Z, going into a convolution
			nn.Conv2d(1, 8, kernel_size=5, stride=1, padding=2),
			nn.BatchNorm2d(8),
			nn.ReLU(True),
			nn.Dropout2d(p=0.1),
			
			nn.Conv2d(8, 16, kernel_size=5, stride=2, padding=2),
			nn.BatchNorm2d(16),
			nn.ReLU(True),
			nn.Dropout2d(p=0.1),

			nn.Conv2d(16, 32, kernel_size=5, stride=1, padding=2),
			nn.BatchNorm2d(32),
			nn.ReLU(True),
			nn.Dropout2d(p=0.1),

			nn.Conv2d(32, 64, kernel_size=5, stride=2, padding=2),
			nn.BatchNorm2d(64),
			nn.ReLU(True),
			nn.Dropout2d(p=0.2),

			nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
			nn.BatchNorm2d(128),
			nn.ReLU(True)
		)
			
		self.classifier = nn.Sequential(
			nn.Linear(128, 10),
		)
		
	def forward(self, x):   
		x = self.main(x)
		x = torch.mean(x.view(x.size(0), x.size(1), -1), dim=2)  # GAP Layer
		logits = self.classifier(x)
		return logits, x
	

class Mnist_9_200(nn.Module):
	def __init__(self):
		# mnist_9_200
		super(Mnist_9_200, self).__init__()
		self.main = nn.Sequential(
			nn.Flatten(),
			nn.Linear(784,200),
			nn.ReLU(),
			nn.Linear(200,200),
			nn.ReLU(),
			nn.Linear(200,200),
			nn.ReLU(),
			nn.Linear(200,200),
			nn.ReLU(),
			nn.Linear(200,200),
			nn.ReLU(),
			nn.Linear(200,200),
			nn.ReLU(),
			nn.Linear(200,200),
			nn.ReLU(),
			nn.Linear(200,200),
			nn.ReLU()
			# nn.ReLU(),
			# nn.Linear(10,10, bias=False)
		)

		self.classifier = nn.Sequential(nn.Linear(200,10),)

	def forward(self, x):   
		x = self.main(x)
		#x = torch.mean(x.view(x.size(0), x.size(1), -1), dim=2)  # GAP Layer
		logits = self.classifier(x)
		return logits, x


class Mnist_relu_4_1024(nn.Module):
	# mnist_relu_4_1024
	def __init__(self):
		super(Mnist_relu_4_1024,self).__init__()
		self.layer1 = nn.Linear(1*28*28, 1024)
		self.layer2 = nn.Linear(1024, 1024)
		self.layer3 = nn.Linear(1024, 1024)
		self.layer4 = nn.Linear(1024, 10)


	def forward(self,x):
		x = x.view(-1, 1*28*28)
		x = F.relu(self.layer1(x))
		x = F.relu(self.layer2(x))
		x = F.relu(self.layer3(x))
		#features = F.relu(self.layer3(x))
		logits = self.layer4(x)
		return logits, x
	


	

class AlibiCNN(nn.Module):
    def __init__(self):
        super(AlibiCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=2, padding=1)  
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.dropout1 = nn.Dropout(0.3)

   
        self.fc1 = nn.Linear(64 * 14 * 14, 256) 
        self.dropout2 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = self.dropout1(x)

        x = torch.flatten(x, 1) 

        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        logits = self.fc2(x)  # Final layer without activation (will apply softmax in loss function)

        return logits, x


class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2) # (28x28 -> 28x28)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0) # (14x14 -> 10x10)

        # Fully connected layers
        self.fc1 = nn.Linear(16 * 5 * 5, 120) # 16 channels, 5x5 feature maps
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)
        
    def forward(self, x):
        # First convolutional layer + activation + pooling
        x = F.relu(self.conv1(x))       # Output: (6, 28, 28)
        x = F.max_pool2d(x, kernel_size=2, stride=2) # Output: (6, 14, 14)
        
        # Second convolutional layer + activation + pooling
        x = F.relu(self.conv2(x))       # Output: (16, 10, 10)
        x = F.max_pool2d(x, kernel_size=2, stride=2) # Output: (16, 5, 5)
        
        # Flatten the tensor for the fully connected layers
        x = x.view(-1, 16 * 5 * 5)     # Output: (batch_size, 16*5*5)
        
        # Fully connected layers + activations
        x = F.relu(self.fc1(x))        # Output: (batch_size, 120)
        x = F.relu(self.fc2(x))        # Output: (batch_size, 84)
        logits = self.fc3(x)                # Output: (batch_size, num_classes)
        
        return logits, x
	

class MLP_schut(nn.Module):
    """A single multi-layer perceptron for classification."""

    def __init__(self, n_hidden=80, input_flat_size=28*28, n_classes=10):
        super().__init__()
        self._net = nn.Sequential(  #
            nn.modules.flatten.Flatten(),  #
            nn.Linear(in_features=input_flat_size, out_features=n_hidden),  #
            nn.ReLU(),  #
            nn.BatchNorm1d(num_features=n_hidden),  #
            nn.Linear(in_features=n_hidden, out_features=n_hidden),  #
            nn.ReLU(),  #
            nn.BatchNorm1d(num_features=n_hidden),  #
            
        )
        self.classifier = nn.Linear(in_features=n_hidden, out_features=n_classes)

    def forward(self, x: torch.Tensor):
        x = self._net(x)
        logits = self.classifier(x)
        #probs = F.softmax(x, dim=-1)
        return logits, x
	


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super(ResNet, self).__init__()
        self.in_channels = 16

        # Initial convolution (smaller for MNIST)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Residual layers
        self.layer1 = self._make_layer(block, 16, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 32, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 64, layers[2], stride=2)

        # Adaptive average pooling and fully connected layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)

        return logits, x





# CIFAR models
class CaptumCNN(nn.Module):
    def __init__(self):
        super(CaptumCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()
        self.relu3 = nn.ReLU()
        self.relu4 = nn.ReLU()

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = self.relu3(self.fc1(x))
        x = self.relu4(self.fc2(x))
        logits = self.fc3(x)
        return logits, x
    




class ResNet_small_cifar(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super(ResNet_small_cifar, self).__init__()
        self.in_channels = 16

        # Initial convolution 
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Residual layers
        self.layer1 = self._make_layer(block, 16, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 32, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 64, layers[2], stride=2)

        # Adaptive average pooling and fully connected layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)

        return logits, x



class ResNet_large_cifar(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super(ResNet_large_cifar, self).__init__()
        self.in_channels = 32

        # Initial convolution 
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU(inplace=True)

        # Residual layers
        self.layer1 = self._make_layer(block, 32, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 64, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 128, layers[2], stride=2)

        # Adaptive average pooling and fully connected layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)

        return logits, x


	

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        # Input size: [batch, 3, 32, 32]
        # Output size: [batch, 3, 32, 32]
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 12, 4, stride=2, padding=1),            # [batch, 12, 16, 16]
            nn.ReLU(),
            nn.Conv2d(12, 24, 4, stride=2, padding=1),           # [batch, 24, 8, 8]
            nn.ReLU(),
            nn.Conv2d(24, 48, 4, stride=2, padding=1),           # [batch, 48, 4, 4]
            nn.ReLU(),
# 			nn.Conv2d(48, 96, 4, stride=2, padding=1),           # [batch, 96, 2, 2]
#             nn.ReLU(),
        )
        self.decoder = nn.Sequential(
#             nn.ConvTranspose2d(96, 48, 4, stride=2, padding=1),  # [batch, 48, 4, 4]
#             nn.ReLU(),
            nn.ConvTranspose2d(48, 24, 4, stride=2, padding=1),  # [batch, 24, 8, 8]
            nn.ReLU(),
            nn.ConvTranspose2d(24, 12, 4, stride=2, padding=1),  # [batch, 12, 16, 16]
            nn.ReLU(),
            nn.ConvTranspose2d(12, 3, 4, stride=2, padding=1),   # [batch, 3, 32, 32]
            nn.Sigmoid(),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded







class Generator_Cifar(nn.Module):
    def __init__(self, ngpu, nc=3, nz=100, ngf=64):
        super(Generator_Cifar, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(     nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 2,     ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(    ngf,      nc, kernel_size=1, stride=1, padding=0, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        if input.is_cuda and self.ngpu > 1:
            output = nn.parallel.data_parallel(self.main, input, range(self.ngpu))
        else:
            output = self.main(input)
        return output
	



In [4]:
def compute_ssim(image, counterfactual):
    return structural_similarity(image, counterfactual, data_range=2, multichannel=True)


def get_implausibility(counterfactual, subset, y_train, target, new_pred, method):
    if counterfactual is None:
        return None
    
    else:
        if method == "Captum-MinParamPerturbation":
            subset = subset[y_train == new_pred]
        else:
            subset = subset[y_train == target]
        
        # Flatten the images in the subset and the counterfactual
        flattened_subset = np.stack([x.ravel() for x in subset])  # Shape: (num_images, flattened_image_size)
        flattened_counterfactual = counterfactual.ravel()  # Shape: (flattened_image_size,)
        
        #ssim_scores_list = []
        # Compute ssim score between each subset and the explanation
        #for i in range(flattened_subset.shape[0]):
        #    ssim_score = structural_similarity(flattened_subset[i], flattened_counterfactual, data_range=2, multichannel=True)
        #    ssim_scores_list.append(ssim_score)

        ssim_scores_list = Parallel(n_jobs=-1)(delayed(compute_ssim)(flattened_subset[i], flattened_counterfactual)
                                       for i in range(flattened_subset.shape[0]))

   
        
        # Compute the mean squared distance (implausibility)
        #print(np.mean(ssim_scores_list))
        return np.mean(ssim_scores_list)




def get_l1(counterfactual, original):
    if counterfactual is None:
        return None
    else:
        return np.sum(np.abs(original - counterfactual))

def get_l2(counterfactual, original):
    if counterfactual is None:
        return None
    else:
        return np.linalg.norm(original- counterfactual)

def get_linf(counterfactual,original):
    if counterfactual is None:
        return None
    else:
        return np.max(np.abs(original - counterfactual))

def get_correctness(counterfactual, target, network):

    if network(counterfactual) == target:
        return True
    return False

def correctness(row):
    if row['Target Label'] == row['New Prediction']: 
        val = 1
    else:
        val = 0
    return val


def weak_correctness(row):
    #### Change code #### --> at least 1 explanation/decision boundary for that instance is correct
    if row['Target Label'] == row['New Prediction']: 
        val = 1
    else:
        val = 0
    return val


def get_IM1(counterfactual, aes, og_prediction, new_prediction):
    """
    Calculate the IM1 metric for CIFAR-10.
    
    Args:
    - counterfactual: The counterfactual image, expected to be normalized between [-1, 1].
    - aes: A list or dictionary of autoencoders for each class.
    - og_prediction: Original class prediction index.
    - new_prediction: New class prediction index after applying the counterfactual.
    
    Returns:
    - IM1 metric
    """
    if counterfactual is None:
        return None
    else:
        # Rescale the counterfactual from [-1, 1] to [0, 1]
        explanation_img = (counterfactual * 0.5) + 0.5
        
        # Ensure the image is a tensor and float type
        explanation_img = torch.from_numpy(explanation_img).float()
        
        
        # Reshape for CIFAR-10 (3 channels, 32x32 size)
        #explanation_img = explanation_img.reshape(-1, 3, 32, 32)
        
        # Pass through the appropriate autoencoder based on predictions
        
        o_recon = aes[og_prediction](explanation_img).flatten()
        e_recon = aes[int(new_prediction)](explanation_img).flatten()
        
        # Calculate the errors
        o_error = sum((o_recon.detach().numpy() - explanation_img.detach().numpy().flatten())**2)
        e_error = sum((e_recon.detach().numpy() - explanation_img.detach().numpy().flatten())**2)
        
        # Return the IM1 metric
        return e_error / o_error


def get_IM2(counterfactual, aes, aefull, new_prediction):
    """
    Calculate the IM2 metric for CIFAR-10.
    
    Args:
    - counterfactual: The counterfactual image, expected to be normalized between [-1, 1].
    - aes: A list or dictionary of autoencoders for each class.
    - aefull: The full autoencoder for all classes.
    - new_prediction: New class prediction index after applying the counterfactual.
    
    Returns:
    - IM2 metric
    """
    if counterfactual is None:
        return None
    else:
        # Rescale the counterfactual from [-1, 1] to [0, 1]
        explanation_img = (counterfactual * 0.5) + 0.5
        
        # Ensure the image is a tensor and float type
        explanation_img = torch.from_numpy(explanation_img).float()
        
        # Reshape for CIFAR-10 (3 channels, 32x32 size)
        #explanation_img = explanation_img.reshape(-1, 3, 32, 32)
        
        # Pass the image through the full autoencoder
        all_recon = aefull(explanation_img).flatten().detach().numpy()
        
        # Pass the image through the class-specific autoencoder
        e_recon = aes[int(new_prediction)](explanation_img).flatten().detach().numpy()
        
        # Calculate the L1 norm of the explanation image
        x_l1_norm = float(torch.sum(torch.abs(explanation_img.flatten())))
        
        # Return the IM2 metric
        return sum((e_recon - all_recon)**2) / x_l1_norm



def get_prediction(image, model, device = 'cpu'):
    if isinstance(image, np.ndarray):
        transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
        image = transform(image).unsqueeze(0)  # Convert to tensor and add batch dimension
    
    image = image.to(device, dtype=torch.float32)
    model = model.to(device)

    # Get the model's prediction
    with torch.no_grad():  # No gradients needed for inference
        output = model(image)[0].argmax()  
    
    return int(output) 





def compute_morans_i(image, kernel_name, size):
        """
        TODO
        :param info:
        :return:
        """
        
        if image is None:
            return None
        else:
            if isinstance(image, torch.Tensor):
                pass
            else:
                image = torch.from_numpy(image)


            grayscale_transform = transforms.Grayscale(num_output_channels=1)
            image = grayscale_transform(image)
            
            ROOK_KERNEL = torch.Tensor([
                                    [0, 1, 0],
                                    [1, 0, 1],
                                    [0, 1, 0],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            BISHOP_KERNEL = torch.Tensor([
                                    [1, 0, 1],
                                    [0, 0, 0],
                                    [1, 0, 1],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            QUEEN_KERNEL = torch.Tensor([
                                    [1, 1, 1],
                                    [1, 0, 1],
                                    [1, 1, 1],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            VERT_KERNEL = torch.Tensor([
                                    [0, 1, 0],
                                    [0, 0, 0],
                                    [0, 1, 0],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            HOR_KERNEL = torch.Tensor([
                                    [0, 0, 0],
                                    [1, 0, 1],
                                    [0, 0, 0],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            KERNEL_BY_NAME = {'rook': ROOK_KERNEL,'bishop': BISHOP_KERNEL,'queen': QUEEN_KERNEL,'vert': VERT_KERNEL,'hor': HOR_KERNEL}
        
            # Make sure the chosen kernel is of shape (input_channels, output_channels, width, height) = (1, 1, 28, 28)
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            kernel = KERNEL_BY_NAME[kernel_name].to(device)
            assert kernel.shape == (1, 1, 3, 3)

         
            

            image_shape = (size, size)  #  make variable. currently hardcoded for mnist.

            # Compute W
            num_neighbours = F.conv2d(torch.ones((1, 1) + image_shape, device=device),
                                    kernel,
                                    padding=1).sum()

            # Normalize the nodes' weights and reshape them to an image
            # Unsqueeze to make a channel dimension (with only one channel)
            # Unsqueeze again to make a minibatch dimension (with only one element)
            norm_weights = (image - torch.mean(image)).reshape(image_shape).unsqueeze(0).unsqueeze(0) 
            
            # Compute the numerator of the fraction
        
            numerator = (F.conv2d(norm_weights,kernel,padding=1) * norm_weights).sum()
            numerator *= (size*size)
            # Compute the denominator of the fraction
            denominator = (norm_weights * norm_weights).sum()
            denominator *= num_neighbours
            # Obtain Moran's I
            I = numerator / denominator
            #print("Moran's I is: ", I.item())
            # Value generally falls between [-1, 1]. Rescale to proper regularization term
            #morans_i = torch.abs(I - 1)

            #morans_loss = morans_i_strength * morans_i #this term can be added to the loss function (with a certain strength) such that the network is optimized to have a high autocorrelation value. 

            return I.item()



In [5]:
def get_cifar():
    # Define the transform to convert images to tensors and normalize to [-1, 1]
    transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])


    # Download and load the training and test datasets with transformations
    train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

    # Extract transformed data and labels by iterating over the datasets
    x_train = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))]).numpy()
    y_train = torch.tensor([train_dataset[i][1] for i in range(len(train_dataset))]).numpy()
    x_test = torch.stack([test_dataset[i][0] for i in range(len(test_dataset))]).numpy()
    y_test = torch.tensor([test_dataset[i][1] for i in range(len(test_dataset))]).numpy()

    return x_train, y_train, x_test, y_test

    
def load_image(image_path):
    return torch.load(image_path) # Assuming images are stored as torch tensors


def load_model(model_path, name):
    if name == CaptumCNN:
        model = name()
        model.load_state_dict(torch.load(model_path))
        
    elif name == ResNet_small_cifar:
         model = name(BasicBlock, [1, 1, 1], num_classes=10)
         model.load_state_dict(torch.load(model_path))
         
    elif name == ResNet_large_cifar:
        model = name(BasicBlock, [2, 2, 2], num_classes=10)
        model.load_state_dict(torch.load(model_path))
    
    else:
        print('add alternative model here')

    model.eval()
    return model
    
def load_autoencoders():
	aes = list()
	for i in range(10):
		model = Autoencoder()
		model.load_state_dict(torch.load('CIFAR_autoencoders/ae_' + str(i) + '.pth', map_location='cpu'))
		model.eval()
		aes.append(model)
	ae_full = Autoencoder()
	ae_full.load_state_dict(torch.load('CIFAR_autoencoders/ae_full.pth', map_location='cpu'))
	ae_full.eval()
	return aes, ae_full


In [7]:
network_list = ["cifar_captum_cnn_output",  "cifar_resnet8_output", "cifar_resnet18_output"]
base_models = "models/cifar"
method_list = ["PIECE", "alibi-CF", "alibi-Proto-CF", "C-Min-Edit", "Captum-MinParamPerturbation", "Min-Edit"]

network_loader = {
    "cifar_captum_cnn_output":[CaptumCNN, "models/cifar/cifar_captum_cnn.pt"],
 "cifar_resnet8_output": [ResNet_small_cifar, "models/cifar/cifar_resnet8.pt"],
 "cifar_resnet18_output": [ResNet_large_cifar, "models/cifar/cifar_resnet18.pt"],

}

captum = "output_Captum_cifar_data.csv"
alibi = "output_Alibi_cifar_data.csv"
piece = "output_PIECE_cifar_data.csv"

# Loading AEs for IM1 and IM2 metrics
aes, ae_full = load_autoencoders()

df = pd.DataFrame(columns=["network", "image", "original_label", "original_predicted_label", "predicted_label", "target", "method", "IM1", "IM2", "correctness", "l1", "l2", "linf", "implausibility", "morans_i_original", "morans_i_explanation", "morans_i_original_minus_explanation", "optim_time"])
x_train, y_train, x_test, y_test  = get_cifar()

for network in network_list:
    print("-------------",network, "-------------")
    network_path = os.path.join(base_models, network).replace("\\", "/")
    net_info = network_loader[network]
    model = load_model(net_info[1], net_info[0])
    captum_csv = pd.read_csv(network + "/" + captum)
    alibi_csv = pd.read_csv(network+ "/" + alibi)
    piece_csv = pd.read_csv(network + "/" + piece)
    # align and rename instances (e.g 12 becomes 11, etc. )
    piece_csv["Instance"] -= 1
    frames = [piece_csv, alibi_csv, captum_csv]
    methods_csv = pd.concat(frames)
    methods_csv.reset_index(drop=True)
    # drop instances where we only have data for some of the methods (a bug in one of the scripts causes this) 
    methods_csv = methods_csv[methods_csv['Instance'] != -1]
    methods_csv = methods_csv[methods_csv['Instance'] != 100]
    methods_csv.reset_index(drop=True)
    methods_csv['correctness'] = methods_csv.apply(correctness, axis=1)

    # Check NaN values --> captum has NaNs in target, which is okay. alibi-... has NaNs if no valid explanation was found.
    rows_with_nan = methods_csv[methods_csv.isnull().any(axis=1)]
    #print(rows_with_nan)

    for method in method_list:
        print("**",method, "**")
        method_path = os.path.join(network, method).replace("\\", "/")
        # Iterate through the methods
        filtered_df = methods_csv[methods_csv['Name'] == method]
        for index, row in filtered_df.iterrows():
            #method_path = os.path.join(network, method).replace("\\", "/")
            i = row['Instance']
            t = row['Target Label']
            p = row['New Prediction']
            #print("This is instance " + str(int(i)) + " with original label " + str(int(y_original)))


            image_original = torch.load(network +"/original/instance_" + str(int(i)+1) +".pt").detach().numpy()
        

            if method == 'PIECE' or method == 'Min-Edit' or method == 'C-Min-Edit':
                new_i = i + 1
                #print("Loading:", method_path +"/instance_" + str(int(i)) + "_target_"+ str(int(t)) +".pt")
                image = torch.load(method_path + "/instance_" + str(int(new_i)) + "_target_"+ str(int(t)) +".pt").detach().numpy()
                
              

            elif method == 'Captum-MinParamPerturbation':
                if pd.isna(p):
                    image = None
                #print("Loading:", method_path +"/instance_" + str(int(i)) + "_target_"+ str(int(p)) +".pt")
                else:
                    image = torch.load(method_path + "/instance_" + str(int(i)) + "_target_"+ str(int(p)) +".pt").detach().numpy()

            else:
                #print("Loading:", method_path +"/instance_" + str(int(i)) + "_target_"+ str(int(t)) +".pt")
                if pd.isna(p):
                    image = None
                else:
                    image = torch.load(method_path +"/instance_" + str(int(i)) + "_target_"+ str(int(t)) +".pt").detach().numpy()
                    #print("min val image", image.min())

            # fix this because I messed up 1 line of code in the alibi cifar experiments
            y_original = y_test[int(i)]

            if image is None:
                image_diff = None
            else:
                if image.shape == (1, 32, 32, 3):
                    image = np.transpose(image, (0, 3, 1, 2))
                else:
                    pass
                image_diff = np.abs(image_original - image)



            # Create a dictionary to store results

            

            
            temp = {
                "network": network,
                "image": i,
                "original_label": y_original,
                "original_predicted_label": int(row['Original Prediction']),
                "predicted_label": p,
                "target": t,
                "method": method,
                "optim_time": row['optim_time'],
                "IM1": get_IM1(image, aes, int(row['Original Prediction']), p),
                "IM2": get_IM2(image, aes, ae_full, p), 
                "correctness": int(row['correctness']), 
                "l1": get_l1(image, image_original),
                "l2": get_l2(image, image_original),
                "linf": get_linf(image, image_original), 
                "implausibility": get_implausibility(image, x_train, y_train, t, p, method),
                "morans_i_original": compute_morans_i(image_original, "queen", 32), 
                "morans_i_explanation": compute_morans_i(image, "queen", 32),
                "morans_i_original_minus_explanation": compute_morans_i(image_diff, "queen", 32)
                
            }
            df_temp = pd.DataFrame([temp])
            df = pd.concat([df, df_temp], ignore_index=True)

#df.to_csv("temp.csv") 
df.head()           


Files already downloaded and verified
Files already downloaded and verified
------------- cifar_captum_cnn_output -------------
** PIECE **
** alibi-CF **
** alibi-Proto-CF **
** C-Min-Edit **
** Captum-MinParamPerturbation **
** Min-Edit **
------------- cifar_resnet8_output -------------
** PIECE **
** alibi-CF **
** alibi-Proto-CF **
** C-Min-Edit **
** Captum-MinParamPerturbation **
** Min-Edit **
------------- cifar_resnet18_output -------------
** PIECE **
** alibi-CF **
** alibi-Proto-CF **
** C-Min-Edit **
** Captum-MinParamPerturbation **
** Min-Edit **


,network,image,original_label,original_predicted_label,predicted_label,target,method,IM1,IM2,correctness,l1,l2,linf,implausibility,morans_i_original,morans_i_explanation,morans_i_original_minus_explanation,optim_time
0,cifar_captum_cnn_output,0.0,3,2,2.0,0.0,PIECE,1.000000,0.001411,0,847.862305,19.216583,1.150527,1232.365845,0.726693,0.852652,0.442470,35.495933
1,cifar_captum_cnn_output,0.0,3,2,1.0,1.0,PIECE,0.942480,0.001729,1,990.313599,22.043844,1.381147,1079.798218,0.726693,0.804315,0.494868,16.772949
2,cifar_captum_cnn_output,0.0,3,2,3.0,3.0,PIECE,1.104398,0.001238,1,769.048706,17.353600,1.120134,947.987671,0.726693,0.866390,0.430576,0.106369
3,cifar_captum_cnn_output,0.0,3,2,3.0,4.0,PIECE,1.145602,0.001186,0,790.880554,17.676945,1.101691,659.654785,0.726693,0.876644,0.419368,35.819453
4,cifar_captum_cnn_output,0.0,3,2,3.0,5.0,PIECE,1.063841,0.001311,0,800.288696,18.072105,1.090838,882.575928,0.726693,0.816604,0.428839,35.985261


In [8]:
df['correctness_ratio'] = df.groupby(['network', 'method', 'image'])['correctness'].transform('sum') / 9

In [9]:
# compute weak correctness 
df['weak_correctness'] = df.groupby(['network', 'method', 'image'])['correctness'].transform('max')

In [12]:
df[df['method'] == 'PIECE'].head(50)

,network,image,original_label,original_predicted_label,predicted_label,target,method,IM1,IM2,correctness,l1,l2,linf,implausibility,morans_i_original,morans_i_explanation,morans_i_original_minus_explanation,optim_time,correctness_ratio,weak_correctness
0,cifar_captum_cnn_output,0.0,3,2,2.0,0.0,PIECE,1.000000,0.001411,0,847.862305,19.216583,1.150527,1232.365845,0.726693,0.852652,0.442470,35.495933,0.555556,1
1,cifar_captum_cnn_output,0.0,3,2,1.0,1.0,PIECE,0.942480,0.001729,1,990.313599,22.043844,1.381147,1079.798218,0.726693,0.804315,0.494868,16.772949,0.555556,1
2,cifar_captum_cnn_output,0.0,3,2,3.0,3.0,PIECE,1.104398,0.001238,1,769.048706,17.353600,1.120134,947.987671,0.726693,0.866390,0.430576,0.106369,0.555556,1
3,cifar_captum_cnn_output,0.0,3,2,3.0,4.0,PIECE,1.145602,0.001186,0,790.880554,17.676945,1.101691,659.654785,0.726693,0.876644,0.419368,35.819453,0.555556,1
4,cifar_captum_cnn_output,0.0,3,2,3.0,5.0,PIECE,1.063841,0.001311,0,800.288696,18.072105,1.090838,882.575928,0.726693,0.816604,0.428839,35.985261,0.555556,1
5,cifar_captum_cnn_output,0.0,3,2,6.0,6.0,PIECE,1.525084,0.001070,1,767.673340,17.359900,1.113133,740.483521,0.726693,0.840386,0.414172,2.639458,0.555556,1
6,cifar_captum_cnn_output,0.0,3,2,7.0,7.0,PIECE,0.823879,0.001012,1,811.518799,18.568460,1.113674,1050.838379,0.726693,0.907092,0.447738,16.167049,0.555556,1
7,cifar_captum_cnn_output,0.0,3,2,2.0,8.0,PIECE,1.000000,0.001453,0,824.388062,18.428953,1.103052,989.676819,0.726693,0.808934,0.387515,35.850888,0.555556,1
8,cifar_captum_cnn_output,0.0,3,2,9.0,9.0,PIECE,0.830469,0.000676,1,789.304382,17.836809,1.170924,1138.121338,0.726693,0.866697,0.451956,1.792601,0.555556,1
9,cifar_captum_cnn_output,1.0,8,1,0.0,0.0,PIECE,1.278599,0.002502,1,925.099670,22.646946,1.533533,1488.650879,0.927461,0.906493,0.581683,0.939323,0.333333,1


In [13]:
# add moran's i metric = |MI(x) - MI(x')|

df['morans_i_difference'] = (df['morans_i_original'] - df['morans_i_explanation']).abs()

In [14]:
df[df['method'] == 'PIECE'].head()

,network,image,original_label,original_predicted_label,predicted_label,target,method,IM1,IM2,correctness,...,l2,linf,implausibility,morans_i_original,morans_i_explanation,morans_i_original_minus_explanation,optim_time,correctness_ratio,weak_correctness,morans_i_difference
0,cifar_captum_cnn_output,0.0,3,2,2.0,0.0,PIECE,1.000000,0.001411,0,...,19.216583,1.150527,1232.365845,0.726693,0.852652,0.442470,35.495933,0.555556,1,0.125959
1,cifar_captum_cnn_output,0.0,3,2,1.0,1.0,PIECE,0.942480,0.001729,1,...,22.043844,1.381147,1079.798218,0.726693,0.804315,0.494868,16.772949,0.555556,1,0.077622
2,cifar_captum_cnn_output,0.0,3,2,3.0,3.0,PIECE,1.104398,0.001238,1,...,17.353600,1.120134,947.987671,0.726693,0.866390,0.430576,0.106369,0.555556,1,0.139697
3,cifar_captum_cnn_output,0.0,3,2,3.0,4.0,PIECE,1.145602,0.001186,0,...,17.676945,1.101691,659.654785,0.726693,0.876644,0.419368,35.819453,0.555556,1,0.149951
4,cifar_captum_cnn_output,0.0,3,2,3.0,5.0,PIECE,1.063841,0.001311,0,...,18.072105,1.090838,882.575928,0.726693,0.816604,0.428839,35.985261,0.555556,1,0.089911


In [15]:
import glob
import os
import re

In [16]:
# Add column to confirm whether or not it timed out (PIECE). Regarding Captum and Alibi methods, if there is a NaN then it timed out. 
# Need .out files for this ... this will be painful ...

timeout_text = "Explanation is invalid! Returning incorrect explanation."





timeout_explanation_count = {
    'network': [],
    'image': [],
    'method': [],
    'target': [],
    'timeout': []
}

next_line = 0

for network in network_list:
    print("-------------",network, "-------------")
    out_file = glob.glob(os.path.join(network, '*.out'))
    print(out_file)
    with open(out_file[0], 'r') as file:
        for line in file:
            if line.startswith(timeout_text):
                next_line = 2
                continue

            elif next_line == 2 and line.startswith("   Instance"):
                next_line = 1
                continue

            elif next_line == 1:
                split_list = line.split()
                #print(split_list)
                timeout_explanation_count['network'].append(network)
                timeout_explanation_count['image'].append(float(split_list[1]))
                timeout_explanation_count['target'].append(float(split_list[4]))
                timeout_explanation_count['method'].append(split_list[2])
                timeout_explanation_count['timeout'].append(int(1))
                next_line = 0
                continue

            else:
                continue


            

            
                


df_timeouts = pd.DataFrame(timeout_explanation_count)
df_timeouts.head(20)


------------- cifar_captum_cnn_output -------------
['cifar_captum_cnn_output\\piece_cifar_captum_cnn_test.out']
------------- cifar_resnet8_output -------------
['cifar_resnet8_output\\piece_cifar_resnet8_test.out']
------------- cifar_resnet18_output -------------
['cifar_resnet18_output\\piece_cifar_resnet18_test.out']


,network,image,method,target,timeout
0,cifar_captum_cnn_output,0.0,PIECE,0.0,1
1,cifar_captum_cnn_output,0.0,PIECE,1.0,1
2,cifar_captum_cnn_output,0.0,PIECE,2.0,1
3,cifar_captum_cnn_output,0.0,Min-Edit,2.0,1
4,cifar_captum_cnn_output,0.0,PIECE,3.0,1
5,cifar_captum_cnn_output,0.0,PIECE,4.0,1
6,cifar_captum_cnn_output,0.0,PIECE,5.0,1
7,cifar_captum_cnn_output,0.0,Min-Edit,5.0,1
8,cifar_captum_cnn_output,0.0,PIECE,6.0,1
9,cifar_captum_cnn_output,0.0,Min-Edit,6.0,1


In [17]:
df

,network,image,original_label,original_predicted_label,predicted_label,target,method,IM1,IM2,correctness,...,l2,linf,implausibility,morans_i_original,morans_i_explanation,morans_i_original_minus_explanation,optim_time,correctness_ratio,weak_correctness,morans_i_difference
0,cifar_captum_cnn_output,0.0,3,2,2.0,0.0,PIECE,1.000000,0.001411,0,...,19.216583,1.150527,1232.365845,0.726693,0.852652,0.442470,35.495933,0.555556,1,0.125959
1,cifar_captum_cnn_output,0.0,3,2,1.0,1.0,PIECE,0.942480,0.001729,1,...,22.043844,1.381147,1079.798218,0.726693,0.804315,0.494868,16.772949,0.555556,1,0.077622
2,cifar_captum_cnn_output,0.0,3,2,3.0,3.0,PIECE,1.104398,0.001238,1,...,17.353600,1.120134,947.987671,0.726693,0.866390,0.430576,0.106369,0.555556,1,0.139697
3,cifar_captum_cnn_output,0.0,3,2,3.0,4.0,PIECE,1.145602,0.001186,0,...,17.676945,1.101691,659.654785,0.726693,0.876644,0.419368,35.819453,0.555556,1,0.149951
4,cifar_captum_cnn_output,0.0,3,2,3.0,5.0,PIECE,1.063841,0.001311,0,...,18.072105,1.090838,882.575928,0.726693,0.816604,0.428839,35.985261,0.555556,1,0.089911
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13795,cifar_resnet18_output,99.0,7,7,4.0,4.0,Min-Edit,1.155196,0.001008,1,...,12.596155,1.075901,1569.442993,0.913938,0.950192,0.493663,3.106606,1.0,1,0.036254
13796,cifar_resnet18_output,99.0,7,7,5.0,5.0,Min-Edit,1.192380,0.000817,1,...,12.384102,1.065520,1764.932983,0.913938,0.952313,0.502398,2.070337,1.0,1,0.038375
13797,cifar_resnet18_output,99.0,7,7,6.0,6.0,Min-Edit,1.517860,0.001048,1,...,12.554259,1.031879,1765.361206,0.913938,0.954944,0.522287,4.012781,1.0,1,0.041006
13798,cifar_resnet18_output,99.0,7,7,8.0,8.0,Min-Edit,1.248493,0.001082,1,...,12.653435,0.995528,1331.796997,0.913938,0.954358,0.525196,6.139853,1.0,1,0.040420


In [18]:
merged_df = pd.merge(df, df_timeouts, on=['network', 'image', 'method', 'target'], how='left')

In [19]:
merged_df[merged_df['method'] == 'PIECE']

,network,image,original_label,original_predicted_label,predicted_label,target,method,IM1,IM2,correctness,...,linf,implausibility,morans_i_original,morans_i_explanation,morans_i_original_minus_explanation,optim_time,correctness_ratio,weak_correctness,morans_i_difference,timeout
0,cifar_captum_cnn_output,0.0,3,2,2.0,0.0,PIECE,1.000000,0.001411,0,...,1.150527,1232.365845,0.726693,0.852652,0.442470,35.495933,0.555556,1,0.125959,1.0
1,cifar_captum_cnn_output,0.0,3,2,1.0,1.0,PIECE,0.942480,0.001729,1,...,1.381147,1079.798218,0.726693,0.804315,0.494868,16.772949,0.555556,1,0.077622,1.0
2,cifar_captum_cnn_output,0.0,3,2,3.0,3.0,PIECE,1.104398,0.001238,1,...,1.120134,947.987671,0.726693,0.866390,0.430576,0.106369,0.555556,1,0.139697,1.0
3,cifar_captum_cnn_output,0.0,3,2,3.0,4.0,PIECE,1.145602,0.001186,0,...,1.101691,659.654785,0.726693,0.876644,0.419368,35.819453,0.555556,1,0.149951,1.0
4,cifar_captum_cnn_output,0.0,3,2,3.0,5.0,PIECE,1.063841,0.001311,0,...,1.090838,882.575928,0.726693,0.816604,0.428839,35.985261,0.555556,1,0.089911,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10095,cifar_resnet18_output,99.0,7,7,4.0,4.0,PIECE,1.157029,0.001025,1,...,0.928210,1587.207153,0.913938,0.953688,0.511577,6.384110,1.0,1,0.039750,NaN
10096,cifar_resnet18_output,99.0,7,7,5.0,5.0,PIECE,1.186544,0.000820,1,...,1.071652,1771.175659,0.913938,0.952589,0.506447,2.265709,1.0,1,0.038651,NaN
10097,cifar_resnet18_output,99.0,7,7,6.0,6.0,PIECE,1.512541,0.001031,1,...,0.999936,1765.995361,0.913938,0.955718,0.518294,4.400131,1.0,1,0.041780,NaN
10098,cifar_resnet18_output,99.0,7,7,8.0,8.0,PIECE,1.269933,0.001144,1,...,0.925022,1350.563965,0.913938,0.951858,0.511908,10.476797,1.0,1,0.037919,NaN


In [20]:
merged_df.loc[
    ((merged_df['method'] == 'PIECE') | (merged_df['method'] == 'C-Min-Edit') | (merged_df['method'] == 'Min-Edit')) & (merged_df['timeout'].isna()), 
    'timeout'
] = 0 

In [21]:
merged_df[merged_df['method'] == 'PIECE']

,network,image,original_label,original_predicted_label,predicted_label,target,method,IM1,IM2,correctness,...,linf,implausibility,morans_i_original,morans_i_explanation,morans_i_original_minus_explanation,optim_time,correctness_ratio,weak_correctness,morans_i_difference,timeout
0,cifar_captum_cnn_output,0.0,3,2,2.0,0.0,PIECE,1.000000,0.001411,0,...,1.150527,1232.365845,0.726693,0.852652,0.442470,35.495933,0.555556,1,0.125959,1.0
1,cifar_captum_cnn_output,0.0,3,2,1.0,1.0,PIECE,0.942480,0.001729,1,...,1.381147,1079.798218,0.726693,0.804315,0.494868,16.772949,0.555556,1,0.077622,1.0
2,cifar_captum_cnn_output,0.0,3,2,3.0,3.0,PIECE,1.104398,0.001238,1,...,1.120134,947.987671,0.726693,0.866390,0.430576,0.106369,0.555556,1,0.139697,1.0
3,cifar_captum_cnn_output,0.0,3,2,3.0,4.0,PIECE,1.145602,0.001186,0,...,1.101691,659.654785,0.726693,0.876644,0.419368,35.819453,0.555556,1,0.149951,1.0
4,cifar_captum_cnn_output,0.0,3,2,3.0,5.0,PIECE,1.063841,0.001311,0,...,1.090838,882.575928,0.726693,0.816604,0.428839,35.985261,0.555556,1,0.089911,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10095,cifar_resnet18_output,99.0,7,7,4.0,4.0,PIECE,1.157029,0.001025,1,...,0.928210,1587.207153,0.913938,0.953688,0.511577,6.384110,1.0,1,0.039750,0.0
10096,cifar_resnet18_output,99.0,7,7,5.0,5.0,PIECE,1.186544,0.000820,1,...,1.071652,1771.175659,0.913938,0.952589,0.506447,2.265709,1.0,1,0.038651,0.0
10097,cifar_resnet18_output,99.0,7,7,6.0,6.0,PIECE,1.512541,0.001031,1,...,0.999936,1765.995361,0.913938,0.955718,0.518294,4.400131,1.0,1,0.041780,0.0
10098,cifar_resnet18_output,99.0,7,7,8.0,8.0,PIECE,1.269933,0.001144,1,...,0.925022,1350.563965,0.913938,0.951858,0.511908,10.476797,1.0,1,0.037919,0.0


In [22]:
merged_df.loc[
    ((merged_df['method'] == 'alibi-CF') | (merged_df['method'] == 'alibi-Proto-CF') | (merged_df['method'] == 'Captum-MinParamPerturbation')) 
    & (merged_df['predicted_label'].isna()), 
    'timeout'
] = 1

In [23]:
merged_df.loc[
    ((merged_df['method'] == 'alibi-CF') | (merged_df['method'] == 'alibi-Proto-CF') | (merged_df['method'] == 'Captum-MinParamPerturbation')) 
    & (merged_df['predicted_label'].notna()), 
    'timeout'
] = 0

In [24]:
merged_df[merged_df['method'] == 'Captum-MinParamPerturbation'][merged_df['timeout'] == 1]

C:\Users\Jasmin\AppData\Local\Temp\ipykernel_14272\2120297690.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  merged_df[merged_df['method'] == 'Captum-MinParamPerturbation'][merged_df['timeout'] == 1]


,network,image,original_label,original_predicted_label,predicted_label,target,method,IM1,IM2,correctness,...,linf,implausibility,morans_i_original,morans_i_explanation,morans_i_original_minus_explanation,optim_time,correctness_ratio,weak_correctness,morans_i_difference,timeout
12810,cifar_resnet18_output,10.0,0,0,NaN,NaN,Captum-MinParamPerturbation,NaN,NaN,0,...,NaN,NaN,0.807651,NaN,NaN,1.346910,0.0,0,NaN,1.0
12821,cifar_resnet18_output,21.0,0,0,NaN,NaN,Captum-MinParamPerturbation,NaN,NaN,0,...,NaN,NaN,0.916465,NaN,NaN,1.344526,0.0,0,NaN,1.0


In [25]:
merged_df.to_csv("cifar_full_results.csv") 